In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')

folder = 'heat_index_files'

filenames = {
    'KAVL': 'KAVL-heatindex-1971-2021.xlsx',
    'KGSO': 'KGSO-heatindex-1971-2021.xlsx',
    'KHSE': 'KHSE-heatindex-1971-2021.xlsx',
    'KILM': 'KILM-heatindex-1971-2021.xlsx',
    'KCLT': 'KLCT-heatindex-1971-2021.xlsx',  # <- This might be a typo, double-check 'KLCT'
    'KRDU': 'KRDU-heat-index-1971-2021.xlsx',
}

dfs = {}

for station, file in filenames.items():
    path = os.path.join(folder, file)
    df = pd.read_excel(path)

    # Check if 'date' column exists
    if 'date' in df.columns:
        df['datetime'] = pd.to_datetime(df['date'], errors='coerce')
    elif all(col in df.columns for col in ['year', 'month', 'day']):
        df['datetime'] = pd.to_datetime(df[['year', 'month', 'day']], errors='coerce')
    else:
        raise KeyError(f"No 'date' or ['year', 'month', 'day'] columns found in {file}")

    df['station'] = station
    df['year'] = df['datetime'].dt.year
    df['month'] = df['datetime'].dt.month
    dfs[station] = df

# Combine all stations
all_data = pd.concat(dfs.values(), ignore_index=True)

# --- Plot 1: Summer High Heat Index (HI_max) ---
plt.figure(figsize=(12, 6))
for station in filenames.keys():
    summer = dfs[station][dfs[station]['month'].isin([6, 7, 8])]
    summer = summer.dropna(subset=['heatindexmax2m'])
    hi_max_avg = summer.groupby('year')['heatindexmax2m'].mean()
    plt.plot(hi_max_avg.index, hi_max_avg.values, label=station)

plt.title('☀️ Average Summer High Heat Index (HI_max, Jun–Aug)')
plt.xlabel('Year')
plt.ylabel('Heat Index (°F)')
plt.legend()
plt.tight_layout()
plt.show()

# --- Plot 2: Max Summer Nighttime Lows (HI_min) ---
plt.figure(figsize=(12, 6))
for station in filenames.keys():
    summer = dfs[station][dfs[station]['month'].isin([6, 7, 8])]
    summer = summer.dropna(subset=['heatindexmin2m'])
    max_lows = summer.groupby('year')['heatindexmin2m'].max()
    plt.plot(max_lows.index, max_lows.values, label=station)

plt.title('🌙 Max Nighttime Heat Index in Summer (HI_min, Jun–Aug)')
plt.xlabel('Year')
plt.ylabel('Heat Index (°F)')
plt.legend()
plt.tight_layout()
plt.show()

# --- Plot 3: Winter Low Temp Trends (HI_min, Dec–Feb) ---
plt.figure(figsize=(12, 6))
for station in filenames.keys():
    df = dfs[station]
    winter = df[df['month'].isin([12, 1, 2])]
    winter = winter.dropna(subset=['heatindexmin2m'])
    winter_avg = winter.groupby('year')['heatindexmin2m'].mean()
    plt.plot(winter_avg.index, winter_avg.values, label=station)

plt.title('❄️ Avg Winter Low Heat Index (HI_min, Dec–Feb)')
plt.xlabel('Year')
plt.ylabel('Heat Index (°F)')
plt.legend()
plt.tight_layout()
plt.show()

KeyError: "No 'date' or ['year', 'month', 'day'] columns found in KAVL-heatindex-1971-2021.xlsx"

In [7]:
print(df.columns)

Index(['datetime', 'heatindexmax2m', 'heatindexmin2m'], dtype='object')
